# E3 — Entrenamiento PCN v2

Continúa el entrenamiento desde `best.pt` (época 97) con mejoras:
- **300 épocas totales** (~200 épocas más)
- **`--w_coarse 1.0`**: mayor peso en la pérdida coarse → el modelo aprende estructura geométrica antes de refinar
- **LR cálido**: warm restart en 5e-5, un solo decay a la mitad en época ~200
- **Batch 64**: la T4 aguanta el doble de batch

⏱️ **Tiempo estimado en T4: ~2 horas**

---
### Antes de ejecutar:
1. Menú → **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**
2. Ajusta las rutas de Drive en la **Celda 3** si las tuyas son distintas

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELDA 2: Clonar repo e instalar dependencias ───────────────
import os

REPO_URL = 'https://github.com/herredoble/TFM-reconstruccion-3D'
REPO_DIR = '/content/TFM'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Directorio de trabajo:', os.getcwd())

!pip install torch numpy matplotlib --quiet
print('Dependencias instaladas.')

In [ ]:
# ── CELDA 3: RUTAS DE DRIVE — AJUSTA AQUÍ SI ES NECESARIO ──────
#
# Para ver tus carpetas en Drive ejecuta en una celda nueva:
#   !ls /content/drive/MyDrive

DRIVE = '/content/drive/MyDrive'

# Checkpoint del primer entrenamiento (best.pt epoch 97)
RUTA_BEST_PT = f'{DRIVE}/Datos_E2_E3/checkpoints_pcn/best.pt'

# Datos de entrenamiento
RUTA_SINTETICO    = f'{DRIVE}/sintetico/roturas'          # 2367 pares sinteticos
RUTA_FB_PROCESADO = f'{DRIVE}/fantastic_breaks_procesado' # 61 pares reales

# Carpeta donde guardar el checkpoint v2 al final
RUTA_SALIDA_DRIVE = f'{DRIVE}/Datos_E2_E3/checkpoints_pcn'

print('Rutas configuradas:')
print(f'  best.pt         : {RUTA_BEST_PT}')
print(f'  sintetico       : {RUTA_SINTETICO}')
print(f'  fantastic_breaks: {RUTA_FB_PROCESADO}')
print(f'  salida Drive    : {RUTA_SALIDA_DRIVE}')

In [ ]:
# ── CELDA 4: Copiar datos y checkpoint a la sesion de Colab ────
import shutil
from pathlib import Path

def copiar_si_falta(src, dst, es_dir=True):
    src, dst = Path(src), Path(dst)
    if dst.exists():
        n = len(list(dst.rglob('*'))) if dst.is_dir() else 1
        print(f'  [OK] ya existe: {dst}  ({n} archivos)')
        return
    if not src.exists():
        print(f'  [ERROR] no encontrado en Drive: {src}')
        print('          Revisa la ruta en la Celda 3.')
        return
    print(f'  Copiando {src.name} ...', end=' ')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if es_dir:
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print('listo.')

copiar_si_falta(RUTA_SINTETICO,    'Datos/sintetico/roturas',          es_dir=True)
copiar_si_falta(RUTA_FB_PROCESADO, 'Datos/fantastic_breaks/procesado', es_dir=True)
copiar_si_falta(RUTA_BEST_PT,      'E3/checkpoints/best.pt',           es_dir=False)

print()
print('Resumen de datos disponibles:')
for carpeta in ['Datos/sintetico/roturas', 'Datos/fantastic_breaks/procesado']:
    n = len(list(Path(carpeta).glob('*.npy'))) if Path(carpeta).exists() else 0
    print(f'  {carpeta}: {n} archivos .npy')
print(f'  E3/checkpoints/best.pt: {Path("E3/checkpoints/best.pt").exists()}')

In [ ]:
# ── CELDA 5: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU disponible: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('AVISO: no hay GPU. Ve a Entorno de ejecucion → Cambiar tipo → T4 GPU')

In [ ]:
# ── CELDA 6: ENTRENAR v2 ───────────────────────────────────────
# Deja esta celda corriendo (~2 horas en T4).
# El progreso aparece linea a linea.
#
# Cambios respecto al primer entrenamiento:
#   --epochs 300    → 200 epocas mas desde la 97
#   --lr 5e-5       → warm restart ligero (el v1 termino en 2.5e-5)
#   --lr_decay 100  → un solo decay, en epoca ~200
#   --batch_size 64 → T4 aguanta el doble
#   --w_coarse 1.0  → la clave: mas enfasis en la forma intermedia
#                     corrige el problema de predicciones sin estructura

!python -m E3.train \
    --resume     E3/checkpoints/best.pt \
    --epochs     300 \
    --lr         5e-5 \
    --lr_decay   100 \
    --batch_size 64 \
    --w_coarse   1.0

In [ ]:
# ── CELDA 7: Guardar en Drive ──────────────────────────────────
import shutil, time
from pathlib import Path

Path(RUTA_SALIDA_DRIVE).mkdir(parents=True, exist_ok=True)

ts = time.strftime('%Y%m%d_%H%M')
dst = f'{RUTA_SALIDA_DRIVE}/best_v2_{ts}.pt'
shutil.copy2('E3/checkpoints/best.pt', dst)
print(f'Checkpoint v2 guardado en Drive: {dst}')

In [ ]:
# ── CELDA 8 (opcional): Evaluacion rapida del modelo v2 ────────
# Ejecuta evaluate.py con el nuevo checkpoint para comparar con v1.
# v1: CD=0.077, F-Score=0.019

!python -m E3.evaluate \
    --checkpoint E3/checkpoints/best.pt \
    --salida     E3/resultados_v2

# Copia los resultados a Drive
shutil.copytree('E3/resultados_v2', f'{RUTA_SALIDA_DRIVE}/resultados_v2_{ts}',
                dirs_exist_ok=True)
print(f'Resultados v2 guardados en Drive.')